# Notebook 00 - Cadrage du mini-projet ETL / DWH

## Positionnement 

Le but du projet est de montrer **comment on fait des choix** sur un cas [BOAMP](https://boamp-datadila.opendatasoft.com/explore/dataset/boamp/api/?dataChart=eyJxdWVyaWVzIjpbeyJjb25maWciOnsiZGF0YXNldCI6ImJvYW1wIiwib3B0aW9ucyI6e319LCJjaGFydHMiOlt7ImFsaWduTW9udGgiOnRydWUsInR5cGUiOiJsaW5lIiwiZnVuYyI6IkNPVU5UIiwic2NpZW50aWZpY0Rpc3BsYXkiOnRydWUsImNvbG9yIjoiIzY2YzJhNSJ9XSwieEF4aXMiOiJkYXRlbGltaXRlcmVwb25zZSIsIm1heHBvaW50cyI6IiIsInRpbWVzY2FsZSI6InllYXIiLCJzb3J0IjoiIn1dLCJkaXNwbGF5TGVnZW5kIjp0cnVlLCJhbGlnbk1vbnRoIjp0cnVlfQ%3D%3D), avec :

1. une collecte par batch,
2. une zone de staging,
3. des transformations et contrôles qualité,
4. un mini data warehouse,
5. des KPI utiles,
6. des jobs, un ordonnancement simple et du monitoring.

Dans le monde réel, un ingénieur data ne commence pas par créer 15 dimensions et 12 facts.

Il commence par demander :

- quels KPI le métier veut voir,
- à quelle fréquence,
- avec quel niveau de fiabilité,
- avec quel coût d'implémentation,
- et avec quel grain.

Ici, on choisit donc un **mini DWH KPI-first**.

## KPI retenus

Nous allons répondre à cinq KPI simples :

1. **Nombre d'avis publiés par jour**
2. **Délai moyen de réponse par type de marché**
3. **Top acheteurs par volume d'avis**
4. **Top CPV principaux**
5. **Taux de rejet par batch**

## Pourquoi ces KPI

Ces KPI nous obligent à faire de vrais choix :

- garder la donnée brute,
- normaliser les dates,
- choisir une clé métier stable (`idweb`),
- gérer les CPV multiples,
- isoler les rejets,
- définir le grain de la fact principale,
- distinguer ce qui relève du DWH et ce qui relève de l'exploitation.

## Modèle choisi

### Dimensions retenues

On garde seulement les dimensions nécessaires :

- `dim_date`
- `dim_buyer`
- `dim_market_type`
- `dim_cpv`

### Facts retenues

On garde seulement deux facts :

- `fact_notice_publication`
- `fact_notice_cpv`

## Pourquoi on ne met pas plus de tables

On aurait pu ajouter d'autres dimensions :

- département,
- procédure,
- nature d'avis,
- famille,
- géographie d'exécution,
- lots,
- critères.

Mais ce serait un mauvais choix, car cela alourdirait le projet **sans aider les KPI déjà définis**.

Un data engineer ne modélise pas "tout ce qu'il voit dans la source".  
Il modélise **ce qui sert le besoin**.

## Pipeline 

1. **Collect**
2. **Staging**
3. **Transformations**
4. **Chargement DWH**
5. **OLAP / KPI**
6. **Jobs / monitoring**

## Ce qu'on veut montrer 

À la fin du projet, vous devez comprendre :

- pourquoi on garde le brut,
- pourquoi on sépare `raw`, `std` et `reject`,
- pourquoi le grain d'une fact est critique,
- pourquoi une relation multi-valuée comme `notice -> cpv` ne doit pas casser la fact principale,
- pourquoi les jobs et la surveillance font partie du pipeline,
- pourquoi un dashboard n'est fiable que si l'intégration en amont est solide. 


## Vue d'ensemble de l'architecture

```text
API BOAMP / lots JSON
        ↓
raw_notice + ops_batch_control
        ↓
stg_raw_notice
        ↓
stg_notice_std + stg_notice_cpv + stg_reject_notice
        ↓
tr_notice_kpi_base + tr_notice_cpv
        ↓
dim_date + dim_buyer + dim_market_type + dim_cpv
        ↓
fact_notice_publication + fact_notice_cpv
        ↓
dm_notice_kpi + KPI
        ↓
ops_job_definition + ops_job_run + ops_job_step_run
```

## Pourquoi cette architecture est réaliste

Un pipeline n'est pas juste un transport de fichiers mais un **système de contrôle** avec staging, rejets, chargement et monitoring. 

C'est exactement la logique adoptée ici :

- la couche brute sécurise la traçabilité,
- le staging standardise,
- la couche transformation prépare,
- le DWH structure pour l'analyse,
- les datamarts exposent les KPI,
- la couche ops permet le pilotage du pipeline.
